# 07 · Review Channels

**What this does:** Planning confirms the **channel** of media spend that has no channel yet, for example whether a placement ran as CTV/OTT, programmatic or streaming. Each label looks like `PARENT_CHILD` (e.g. `CTVOTT_AMAZON_FIRETV`); the parent (`CTVOTT`) is the model channel. The pipeline suggests a label for every Prisma placement, but **nothing is used until a person confirms it**.

| | |
|---|---|
| **When to run** | After new media is built (06_Build_Core), to send Planning a workbook; and when Planning sends it back |
| **How long** | About 1 minute (the suggestions are retrained each time a workbook is made) |
| **What it changes** | Only the table `channel_label`, and only when `DRY_RUN = False`. Every change is saved in `title_history` with who, when and why. |
| **What is on the list** | Only titles released in 2023 or later (the MMM's titles). Home Entertainment media only (clients ZUE and ZUV; Focus theatrical media under ZFF and ZFE is not reviewed). Every Prisma placement and every TV and radio network code that still has no channel, grouped by film. The workbook's **Summary** tab shows how many are left per film. |
| **Safe to re-run?** | Yes. Rows that got a channel after the workbook was made are skipped. |

Workbooks live in a Domino dataset. To get there: open this project's **Data** page → **Datasets** → **nbcu_mmm_dev**, then click through **us_hment_mmm → review → channels**. It has three folders:

| Folder | What is in it |
|---|---|
| `issued` | New workbooks to send to Planning |
| `returned` | Where you upload the workbook Planning filled in |
| `imported` | Workbooks already saved (moved here automatically) |

### How to use it
1. **Get a workbook.** With `returned` empty, click **Run ▸ Run All Cells**. Step 2 shows the new file name in `issued`.
2. **Send it to Planning.** Download it from `issued` and send it. In Excel they work through **1 · Placements** (grouped by film; the **How sure** column says *Likely right*, *Closer look* or *New site*) and **2 · TV and radio**. For each row: **Accept**, **Use my answer** (pick the right label from the drop-down, or type a new one), or **Remove placement** with a reason for rows that are not media we model (e.g. $0 added value). Rows they are not sure about stay empty. The workbook's **How to fill this in** tab explains each choice with examples.
3. **Upload it to `returned`** with the **Upload files** button on that folder's page, then **Run All** (a check only). Read **Step 3** (every label that will be saved) and **Step 4** (rows not used). If something needs fixing, fix it in Excel and upload it again.
4. **Save.** If Step 3 is right, set `DRY_RUN = False` and **Run All**. The labels are saved and the workbook moves to `imported`. Set `DRY_RUN = True` again afterwards.

## Step 1 · Settings

In [ ]:
# True  = check only, nothing is changed (always start here)
# False = save the labels in the returned workbook
DRY_RUN = True

## Step 2 · Run

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.channels.review import (changes_table, notes_markdown, quality_markdown, run,
                                          status_markdown)

try:
    result = run(dry_run=DRY_RUN)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — the channel review could not run\n\n`{exc}`\n\nNothing was "
                     "changed. Save this notebook and send it to the pipeline owner."))
    raise

display(status_markdown(result))

## Step 3 · Labels
Every label that will be saved (check only) or that was saved (`DRY_RUN = False`).

In [ ]:
display(changes_table(result['changes']))

## Step 4 · Rows not used
Rows that got a channel after the workbook was made, or that are no longer on the list.

In [ ]:
display(notes_markdown(result['skipped']))

## Step 5 · How good are the suggestions?
Tested each time a workbook is made, on films the model had not seen.

In [ ]:
display(quality_markdown(result))

---
This notebook needs a person to make the decisions, so it is not meant to run as a Domino Job.